# Reproducing Tradewell et al. 2022

Independent reproduction of *Algorithms for Predicting the Probability of
Azoospermia from Follicle Stimulating Hormone* (World J Mens Health
2022;40:600-607) from the authors' CC0 data deposit
(doi:10.7910/DVN/EQFMCM), using `seminal` for the published equations.

Sections:

1. Cohort reconstruction: stated vs reproduced exclusion criteria
2. Table 1
3. Refit of the logistic model on the Miami development cohort
4. External validation in depth: calibration and decision curves
5. Recalibration per site
6. Sensitivity: sample-level vs first-sample-per-man

Sections 3 to 6 are in progress.

## 1. Cohort reconstruction

The Methods list four exclusions: prior vasectomy, Y chromosome microdeletion,
solitary testis, and testosterone or anabolic steroid use in the prior 120
days. Applied as written to the deposit, they give Miami 943 and Chicago 614,
against the paper's 946 and 596. Milan (1,955) matches either way.

The rule that reproduces the paper's sample sizes exactly differs in three
ways, all reverse-engineered from the deposit:

* Y chromosome microdeletion was **not** excluded (2 Miami rows, 3 Chicago rows)
* Absent testis is `Why_Abs` codes 1 to 3 at Miami (code 4 kept) and a
  recorded testis volume of 0 at Chicago, which has no absent-testis column
* Chicago counts sperm concentration below 0.01 million/mL as azoospermia
  (3 cryptozoospermic samples recorded as 0.00001 to 0.001)

`preprocessing.py` implements both as `--exclusions stated` and
`--exclusions reproduced` (the default).

In [ ]:
import subprocess
import pandas as pd

# Build both versions of the processed file
for rule in ["stated", "reproduced"]:
    subprocess.run(
        ["python", "../preprocessing.py", "--exclusions", rule,
         "--raw-dir", "../data/raw", "--out-dir", f"../data/processed/{rule}"],
        check=True, capture_output=True,
    )

paper = pd.DataFrame(
    {"n": [946, 1955, 596, 3497], "azoospermic": [131, 465, 191, 787]},
    index=["miami", "milan", "chicago", "total"],
)

def cohort_counts(rule):
    d = pd.read_parquet(f"../data/processed/{rule}/tradewell_2022.parquet")
    d = d[d["included"]]
    out = d.groupby("site").agg(n=("azoospermia", "size"), azoospermic=("azoospermia", "sum"))
    out.loc["total"] = out.sum()
    return out.loc[["miami", "milan", "chicago", "total"]]

pd.concat(
    {"paper": paper, "stated criteria": cohort_counts("stated"), "reproduced rule": cohort_counts("reproduced")},
    axis=1,
)

## 2. Table 1

Reproduced with `eda_toolkit.generate_table1`, one call per site, with the
paper's reference-range categories (built in `preprocessing.py`). Boundary
values belong to the normal range.

Two cells do not match the printed table and cannot be made to:

* **Milan FSH categories.** The paper reports 1,453 normal / 54 low / 448 high.
  The deposit's Milan sheet carries a precomputed FSH category whose "high"
  group starts at 12.5 IU/L, Milan's own laboratory limit, not the 7.6 stated
  in the Methods. On a single 7.6 cutoff the counts are 1,162 / 54 / 739, and
  the Total column changes accordingly.
* **Chicago testis volume.** The paper reports a median of 14.8 (10.1-17.4) mL
  with a normal spread across the size categories. The deposit holds raw
  ultrasound measurements with a median of 4.5 (4.0-4.8), evidently in cm.
  The authors converted to volume before analysis; the conversion is not
  recoverable from the deposited file.

Everything else matches to the row or within one boundary value.

In [ ]:
from eda_toolkit import generate_table1

df = pd.read_parquet("../data/processed/reproduced/tradewell_2022.parquet")
df = df[df["included"]]

continuous = ["age", "sperm_concentration", "fsh", "lh", "testosterone", "testis_volume"]
categorical = ["azoospermia", "fsh_cat", "lh_cat", "tt_cat", "vol_cat"]

# Rounding per variable, as printed in the paper's Table 1
decimals = {"age": 0, "sperm_concentration": 1, "fsh": 1, "lh": 1, "testosterone": 0, "testis_volume": 0}

# Paper-style row labels and order
labels = {
    "age": "Age (y)",
    "sperm_concentration": "Sperm concentration (mil/mL)",
    "azoospermia = 1": "  Azoospermic",
    "fsh": "FSH (IU/L)",
    "lh": "LH (IU/L)",
    "testosterone": "TT (ng/dL)",
    "testis_volume": "Mean testis size (cm3)",
}
header_rows = ["azoospermia", "azoospermia = 0", "fsh_cat", "lh_cat", "tt_cat", "vol_cat"]
order = [
    "Number of samples", "Age (y)", "Sperm concentration (mil/mL)", "  Azoospermic",
    "FSH (IU/L)", "  Normal (1.5-7.6)", "  Low (<1.5)", "  High (>7.6)",
    "LH (IU/L)", "  Normal (1.7-8.6)", "  Low (<1.7)", "  High (>8.6)",
    "TT (ng/dL)", "  Normal (300-1,000)", "  Low (<300)", "  High (>1,000)",
    "Mean testis size (cm3)", "  Normal (12.5-19)", "  Small (<12.5)", "  Large (>19)",
]

cols = []
for name, g in [
    ("Miami", df[df.site == "miami"]),
    ("Milan", df[df.site == "milan"]),
    ("Chicago", df[df.site == "chicago"]),
    ("Total", df),
]:
    cont, cat = generate_table1(
        g, continuous_cols=continuous, categorical_cols=categorical,
        combine=False, value_counts=True, decimal_places=1,
    )
    cont, cat = cont._df.copy(), cat._df.copy()  # unwrap TableWrapper

    # median (IQR); generate_table1 reports median but not IQR
    q = g[continuous].quantile([0.25, 0.5, 0.75]).T
    cont["Overall"] = [
        f"{q.loc[v, 0.5]:.{decimals[v]}f} "
        f"({q.loc[v, 0.25]:.{decimals[v]}f}-{q.loc[v, 0.75]:.{decimals[v]}f})"
        for v in cont["Variable"]
    ]

    t = pd.concat([cont, cat])[["Variable", "Overall"]].set_index("Variable")["Overall"]
    t.loc["Number of samples"] = f"{len(g):,}"
    cols.append(t.rename(name))

table1 = (
    pd.concat(cols, axis=1)
    .drop(index=header_rows)
    .rename(index=lambda s: labels.get(s, "  " + s.split(" = ", 1)[1] if " = " in s else s))
    .loc[order]
)
table1.index.name = "Variable"
table1

## 3. Refit on the Miami development cohort

*In progress.* Fit the same logistic regression (age, FSH, LH, TT, mean testis
volume) on the 946 Miami rows, compare to Table 3, and report
full-precision coefficients.

## 4. External validation in depth

*In progress.* Calibration-in-the-large and slope, decision curves, and the
effect of Chicago's testis-volume units on the logistic model.

## 5. Recalibration per site

*In progress.*

## 6. Sensitivity: sample-level vs first-sample-per-man

*In progress.*